# ðŸ§¹ Data Cleaning OpenEnv â€” Full Colab Demo

**Real-world OpenEnv environment** where AI agents learn to detect, diagnose,
and fix data quality issues in tabular datasets.

| Task | Difficulty | Steps | What the agent does |
|------|-----------|-------|---------------------|
| Schema Validation | Easy | 1 | Find all issues in a customer CRM export |
| Format Standardization | Medium | 1 | Normalize 5 inconsistently-formatted columns |
| Multi-step Pipeline | Hard | â‰¤8 | Audit â†’ Identify â†’ Fix â†’ Validate |

This notebook is **fully self-contained** â€” runs end-to-end on Colab with no extra files.

## 1. Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn pydantic openai requests nest_asyncio

## 2. Pydantic Models (Observation Â· Action Â· Results)

In [ ]:
from typing import Any, Dict, List, Optional
from enum import Enum
from pydantic import BaseModel, Field

class TaskType(str, Enum):
    SCHEMA_VALIDATION = 'schema_validation'
    STANDARDIZATION   = 'standardization'
    PIPELINE          = 'pipeline'

class IssueType(str, Enum):
    MISSING_REQUIRED = 'missing_required'
    WRONG_TYPE       = 'wrong_type'
    INVALID_RANGE    = 'invalid_range'
    INVALID_FORMAT   = 'invalid_format'
    INVALID_ENUM     = 'invalid_enum'
    DUPLICATE        = 'duplicate'
    OUTLIER          = 'outlier'

class PipelineStep(str, Enum):
    AUDIT    = 'audit'
    IDENTIFY = 'identify'
    FIX      = 'fix'
    VALIDATE = 'validate'

class ColumnStats(BaseModel):
    dtype:         str
    null_count:    int
    unique_count:  int
    sample_values: List[Any]     = Field(default_factory=list)
    min_value:     Optional[Any] = None
    max_value:     Optional[Any] = None

class DataIssue(BaseModel):
    row_index:   int
    column:      str
    issue_type:  IssueType
    description: str = ''
    value:       Optional[Any] = None

class ColumnTransform(BaseModel):
    target_format: str
    regex_from:    Optional[str] = None
    replace_with:  Optional[str] = None
    examples:      List[Dict[str, str]] = []

class FixOperation(BaseModel):
    row_index: int; column: str
    old_value: Optional[Any]; new_value: Optional[Any]
    rationale: str = ''

class DataObservation(BaseModel):
    episode_id:       str
    task_type:        TaskType
    dataset_name:     str
    dataset_sample:   List[Dict[str, Any]] = []
    total_rows:       int = 0
    dataset_schema:   Dict[str, Any] = Field(default_factory=dict, alias='schema')
    column_stats:     Dict[str, ColumnStats] = {}
    current_step:     int = 0
    max_steps:        int = 1
    pipeline_phase:   Optional[PipelineStep] = None
    issues_found:     List[DataIssue] = []
    available_actions: List[str] = []
    reward:           float = 0.0   # step reward (OpenEnv Observation contract)
    done:             bool  = False
    cumulative_reward: float = 0.0
    context:          Dict[str, Any] = {}

class DataAction(BaseModel):
    action_type:       str
    issues:            Optional[List[DataIssue]] = None
    transforms:        Optional[Dict[str, ColumnTransform]] = None
    audit_summary:     Optional[str] = None
    issue_categories:  Optional[List[str]] = None
    fix_operations:    Optional[List[FixOperation]] = None
    validation_report: Optional[str] = None
    issues_remaining:  Optional[int] = None

class RewardBreakdown(BaseModel):
    total: float; components: Dict[str, float] = {}; feedback: str = ''

class StepResult(BaseModel):
    observation: DataObservation; reward: float; done: bool
    metadata:    Dict[str, Any] = {}

class ResetResult(BaseModel):
    observation: DataObservation; episode_id: str

class StateResult(BaseModel):
    episode_id: str; task_type: TaskType; current_step: int; max_steps: int
    pipeline_phase: Optional[PipelineStep]; done: bool
    cumulative_reward: float; action_history: List[Dict[str, Any]] = []

# Rebuild so Pydantic v2 resolves all forward references
for _m in (ColumnStats, DataIssue, ColumnTransform, FixOperation,
           DataObservation, DataAction, RewardBreakdown,
           StepResult, ResetResult, StateResult):
    _m.model_rebuild()

print('Models loaded.')

## 3. Datasets
Three embedded synthetic datasets with **known ground-truth issues** â€” fully deterministic.

In [ ]:
# â”€â”€ Task 1: 30-row customer CRM with 13 known issues â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
SCHEMA_VALIDATION_SCHEMA = {
    'id':         {'type': 'integer',  'required': True,  'unique': True},
    'name':       {'type': 'string',   'required': True,  'min_length': 2},
    'email':      {'type': 'email',    'required': True},
    'phone':      {'type': 'phone_us', 'required': False},
    'age':        {'type': 'integer',  'required': True,  'min': 18, 'max': 120},
    'status':     {'type': 'enum',     'required': True,  'values': ['active','inactive','pending']},
    'created_at': {'type': 'date_iso', 'required': True},
}
SCHEMA_VALIDATION_ISSUES = [
    {'row_index': 2,  'column': 'email',      'issue_type': 'invalid_format',   'description': "Email 'invalid-email' missing '@'"},
    {'row_index': 6,  'column': 'age',        'issue_type': 'invalid_range',    'description': 'Age -5 below min 18'},
    {'row_index': 9,  'column': 'id',         'issue_type': 'duplicate',        'description': 'ID 2 duplicated'},
    {'row_index': 11, 'column': 'email',      'issue_type': 'missing_required', 'description': 'email is null'},
    {'row_index': 14, 'column': 'status',     'issue_type': 'invalid_enum',     'description': "'banned' not in allowed values"},
    {'row_index': 17, 'column': 'age',        'issue_type': 'invalid_range',    'description': 'Age 150 exceeds max 120'},
    {'row_index': 19, 'column': 'name',       'issue_type': 'invalid_format',   'description': 'Empty name'},
    {'row_index': 21, 'column': 'created_at', 'issue_type': 'invalid_format',   'description': 'Invalid month 13'},
    {'row_index': 24, 'column': 'id',         'issue_type': 'duplicate',        'description': 'ID 5 duplicated'},
    {'row_index': 26, 'column': 'phone',      'issue_type': 'invalid_format',   'description': 'Phone not US format'},
    {'row_index': 27, 'column': 'email',      'issue_type': 'invalid_format',   'description': "Email 'test@' malformed"},
    {'row_index': 28, 'column': 'age',        'issue_type': 'wrong_type',       'description': "Age 'twenty' is a string"},
    {'row_index': 29, 'column': 'status',     'issue_type': 'missing_required', 'description': 'status is null'},
]
def _customer_rows():
    rows = [
        {'id':1,  'name':'Alice Smith',   'email':'alice@example.com',  'phone':'(415) 555-0101','age':34,'status':'active',   'created_at':'2022-03-10'},
        {'id':2,  'name':'Bob Jones',     'email':'bob@example.com',    'phone':'(415) 555-0102','age':28,'status':'active',   'created_at':'2022-04-01'},
        {'id':3,  'name':'Clara Wu',      'email':'invalid-email',      'phone':'(415) 555-0103','age':22,'status':'inactive','created_at':'2022-04-15'},
        {'id':4,  'name':'David Lee',     'email':'david@example.com',  'phone':'(415) 555-0104','age':45,'status':'active',   'created_at':'2022-05-01'},
        {'id':5,  'name':'Eva Green',     'email':'eva@example.com',    'phone':'(415) 555-0105','age':31,'status':'pending',  'created_at':'2022-05-12'},
        {'id':6,  'name':'Frank Hall',    'email':'frank@example.com',  'phone':'(415) 555-0106','age':52,'status':'active',   'created_at':'2022-06-01'},
        {'id':7,  'name':'Grace Kim',     'email':'grace@example.com',  'phone':'(415) 555-0107','age':-5,'status':'active',   'created_at':'2022-06-15'},
        {'id':8,  'name':'Hank Moore',    'email':'hank@example.com',   'phone':'(415) 555-0108','age':60,'status':'inactive','created_at':'2022-07-01'},
        {'id':9,  'name':'Iris Chen',     'email':'iris@example.com',   'phone':'(415) 555-0109','age':27,'status':'active',   'created_at':'2022-07-20'},
        {'id':2,  'name':'Jack Brown',    'email':'jack@example.com',   'phone':'(415) 555-0110','age':38,'status':'pending',  'created_at':'2022-08-01'},
        {'id':11, 'name':'Karen Davis',   'email':'karen@example.com',  'phone':'(415) 555-0111','age':44,'status':'active',   'created_at':'2022-08-15'},
        {'id':12, 'name':'Leo Martin',    'email':None,                 'phone':'(415) 555-0112','age':33,'status':'active',   'created_at':'2022-09-01'},
        {'id':13, 'name':'Mia White',     'email':'mia@example.com',    'phone':'(415) 555-0113','age':29,'status':'inactive','created_at':'2022-09-20'},
        {'id':14, 'name':'Noah Taylor',   'email':'noah@example.com',   'phone':'(415) 555-0114','age':55,'status':'active',   'created_at':'2022-10-01'},
        {'id':15, 'name':'Olivia Wilson', 'email':'olivia@example.com', 'phone':'(415) 555-0115','age':36,'status':'banned',   'created_at':'2022-10-15'},
        {'id':16, 'name':'Paul Anderson', 'email':'paul@example.com',   'phone':'(415) 555-0116','age':48,'status':'active',   'created_at':'2022-11-01'},
        {'id':17, 'name':'Quinn Thomas',  'email':'quinn@example.com',  'phone':'(415) 555-0117','age':41,'status':'pending',  'created_at':'2022-11-20'},
        {'id':18, 'name':'Rachel Clark',  'email':'rachel@example.com', 'phone':'(415) 555-0118','age':150,'status':'active',  'created_at':'2022-12-01'},
        {'id':19, 'name':'Sam Lewis',     'email':'sam@example.com',    'phone':'(415) 555-0119','age':25,'status':'inactive','created_at':'2022-12-15'},
        {'id':20, 'name':'',              'email':'noname@example.com', 'phone':'(415) 555-0120','age':30,'status':'active',   'created_at':'2023-01-01'},
        {'id':21, 'name':'Tina Harris',   'email':'tina@example.com',   'phone':'(415) 555-0121','age':37,'status':'active',   'created_at':'2023-01-15'},
        {'id':22, 'name':'Uma Young',     'email':'uma@example.com',    'phone':'(415) 555-0122','age':43,'status':'pending',  'created_at':'2023-13-01'},
        {'id':23, 'name':'Victor King',   'email':'victor@example.com', 'phone':'(415) 555-0123','age':50,'status':'active',   'created_at':'2023-02-01'},
        {'id':24, 'name':'Wendy Scott',   'email':'wendy@example.com',  'phone':'(415) 555-0124','age':39,'status':'inactive','created_at':'2023-02-15'},
        {'id':5,  'name':'Xavier Green',  'email':'xavier@example.com', 'phone':'(415) 555-0125','age':26,'status':'active',   'created_at':'2023-03-01'},
        {'id':26, 'name':'Yara Walker',   'email':'yara@example.com',   'phone':'(415) 555-0126','age':32,'status':'pending',  'created_at':'2023-03-15'},
        {'id':27, 'name':'Zoe Evans',     'email':'zoe@example.com',    'phone':'abc-def-ghij',  'age':58,'status':'active',   'created_at':'2023-04-01'},
        {'id':28, 'name':'Aaron Baker',   'email':'test@',              'phone':'(415) 555-0128','age':47,'status':'active',   'created_at':'2023-04-15'},
        {'id':29, 'name':'Beth Nelson',   'email':'beth@example.com',   'phone':'(415) 555-0129','age':'twenty','status':'inactive','created_at':'2023-05-01'},
        {'id':30, 'name':'Carl Carter',   'email':'carl@example.com',   'phone':'(415) 555-0130','age':63,'status':None,       'created_at':'2023-05-15'},
    ]
    return rows
SCHEMA_VALIDATION_ROWS = _customer_rows()

# â”€â”€ Task 2: 20-row sales records with format inconsistencies â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
STANDARDIZATION_GROUND_TRUTH = {
    'date': ['2023-01-15','2023-02-28','2023-03-10','2023-04-05','2023-05-20',
             '2023-06-14','2023-07-04','2023-08-31','2023-09-12','2023-10-01',
             '2023-11-11','2023-12-25','2023-01-30','2023-02-14','2023-03-21',
             '2023-04-18','2023-05-05','2023-06-30','2023-07-15','2023-08-08'],
    'phone': ['(555) 123-4567','(555) 234-5678','(555) 345-6789','(555) 456-7890',
              '(555) 567-8901','(555) 678-9012','(555) 789-0123','(555) 890-1234',
              '(555) 901-2345','(555) 012-3456','(555) 111-2222','(555) 222-3333',
              '(555) 333-4444','(555) 444-5555','(555) 555-6666','(555) 666-7777',
              '(555) 777-8888','(555) 888-9999','(555) 999-0000','(555) 100-2000'],
    'state': ['CA','NY','TX','FL','WA','IL','OH','PA','GA','NC',
              'MA','AZ','CO','VA','TN','OR','MN','MO','WI','MD'],
    'amount': [1234.56,2500.00,899.99,4500.00,150.00,3200.75,750.50,1100.00,9999.99,450.25,
               2800.00,600.00,1750.50,3300.00,875.25,2150.00,490.00,5500.00,1250.75,675.00],
    'product_code': [f'SKU-{i:03d}' for i in range(1,21)],
}
def _sales_rows():
    dates   = ['2023-01-15','02/28/2023','Mar 10 2023','04/05/23','2023-05-20',
                'Jun 14 2023','07-04-2023','08/31/2023','Sep 12 2023','2023-10-01',
                '11/11/2023','Dec 25 2023','01/30/23','Feb 14 2023','2023-03-21',
                '04/18/2023','May 5 2023','06/30/2023','2023-07-15','Aug 08 2023']
    phones  = ['(555) 123-4567','555-234-5678','+15553456789','5554567890',
                '555.567.8901','(555)678-9012','555 789 0123','(555) 890-1234',
                '555-901-2345','5550123456','(555) 111-2222','555.222.3333',
                '+15553334444','5554445555','555-555-6666','(555) 666-7777',
                '555 777 8888','5558889999','(555)999-0000','+15551002000']
    states  = ['California','New York','TX','Florida','Washington','Illinois','ohio',
                'PA','Georgia','North Carolina','Massachusetts','AZ','colorado',
                'Virginia','Tennessee','OR','Minnesota','Missouri','WI','Maryland']
    amounts = ['$1,234.56','USD 2500.00','899.99','$4,500.00','150','$3,200.75',
                '750.50 USD','1,100.00','$9,999.99','450.25','USD 2800','$600.00',
                '1750.50','$3,300.00','875.25','2,150.00','$490','5500.00 USD','$1,250.75','675']
    skus    = ['SKU-001','sku002','003','SKU 004','SKU-005','sku-006','SKU007',
                'SKU 008','009','SKU-010','sku011','SKU-012','013','SKU 014','SKU-015',
                'sku-016','SKU017','SKU 018','019','SKU-020']
    reps    = ['Alice','Bob','Clara','David','Eva','Frank','Grace','Hank','Iris','Jack',
                'Karen','Leo','Mia','Noah','Olivia','Paul','Quinn','Rachel','Sam','Tina']
    return [{'sale_id':i+1,'rep':reps[i],'date':dates[i],'phone':phones[i],
             'state':states[i],'amount':amounts[i],'product_code':skus[i]} for i in range(20)]
STANDARDIZATION_ROWS = _sales_rows()
STANDARDIZATION_TARGET_FORMATS = {
    'date':         'ISO 8601: YYYY-MM-DD',
    'phone':        'US format: (XXX) XXX-XXXX',
    'state':        'Two-letter uppercase code (e.g. CA)',
    'amount':       'Plain float, 2 decimal places (e.g. 1234.56)',
    'product_code': 'SKU-NNN, zero-padded to 3 digits',
}

# â”€â”€ Task 3: 25-row employee records with 10 issues â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
PIPELINE_SCHEMA = {
    'employee_id':   {'type':'integer', 'required':True,  'unique':True},
    'full_name':     {'type':'string',  'required':True,  'min_length':2},
    'department':    {'type':'enum',    'required':True,
                      'values':['Engineering','Marketing','Sales','Finance','HR','Operations']},
    'salary':        {'type':'float',   'required':True,  'min':20000, 'max':500000},
    'hire_date':     {'type':'date_iso','required':True},
    'vacation_days': {'type':'integer', 'required':True,  'min':0, 'max':60},
    'email':         {'type':'email',   'required':True},
}
PIPELINE_ISSUES = [
    {'row_index':2,  'column':'department',    'issue_type':'invalid_enum',     'description':"'Devops' not allowed"},
    {'row_index':5,  'column':'salary',        'issue_type':'outlier',          'description':'850000 exceeds max 500000'},
    {'row_index':8,  'column':'employee_id',   'issue_type':'duplicate',        'description':'ID 9 duplicated'},
    {'row_index':11, 'column':'hire_date',     'issue_type':'invalid_format',   'description':"'15-06-2021' not ISO"},
    {'row_index':14, 'column':'vacation_days', 'issue_type':'invalid_range',    'description':'-3 below min 0'},
    {'row_index':17, 'column':'salary',        'issue_type':'outlier',          'description':'1200000 exceeds max 500000'},
    {'row_index':19, 'column':'department',    'issue_type':'missing_required', 'description':'null department'},
    {'row_index':21, 'column':'hire_date',     'issue_type':'invalid_format',   'description':"'2021/08/22' uses slashes"},
    {'row_index':23, 'column':'vacation_days', 'issue_type':'invalid_range',    'description':'-7 below min 0'},
    {'row_index':24, 'column':'email',         'issue_type':'invalid_format',   'description':"'notanemail' missing '@'"},
]
PIPELINE_ISSUE_CATEGORIES = ['invalid_enum','outlier','duplicate','invalid_format','invalid_range','missing_required']

def _employee_rows():
    names  = ['Alice Johnson','Bob Smith','Clara Devlin','David Brown','Eva Martinez',
               'Frank Wilson','Grace Lee','Hank Thomas','Iris Chen','Iris Chen',
               'Jack Davis','Karen Miller','Leo Taylor','Mia Anderson','Noah White',
               'Olivia Harris','Paul Clark','Quinn Lewis','Rachel Robinson','Sam Walker',
               'Tina Hall','Uma Young','Victor King','Wendy Wright','Xavier Scott']
    depts  = ['Engineering','Marketing','Devops','Sales','Finance','HR','Operations',
               'Engineering','Marketing','Sales','Finance','HR','Operations','Engineering',
               'Marketing','Sales','Finance','HR','Operations','Engineering',None,
               'Marketing','Sales','Finance','HR']
    sals   = [85000,72000,91000,65000,110000,850000,78000,95000,62000,83000,
               74000,88000,57000,120000,69000,105000,81000,1200000,76000,94000,
               87000,63000,99000,73000,115000]
    hdates = ['2019-03-15','2020-07-01','2018-11-20','2021-02-10','2017-06-05',
               '2022-01-15','2020-09-30','2019-04-22','2021-08-14','2018-12-01',
               '2020-03-18','15-06-2021','2019-07-11','2022-03-05','2018-08-28',
               '2021-11-17','2020-02-24','2019-10-09','2022-06-01','2018-05-14',
               '2021-03-25','2021/08/22','2020-12-07','2019-01-30','2022-09-18']
    vdays  = [15,20,10,25,30,18,22,12,28,16,24,19,11,27,-3,21,17,26,13,23,29,14,-7,20,18]
    emails = [f'{n.split()[0].lower()}.{n.split()[1][0].lower()}@company.com' for n in names]
    emails[24] = 'notanemail'
    rows = []
    for i in range(25):
        eid = i+1 if i != 9 else 9
        rows.append({'employee_id':eid,'full_name':names[i],'department':depts[i],
                     'salary':float(sals[i]),'hire_date':hdates[i],
                     'vacation_days':vdays[i],'email':emails[i]})
    return rows
PIPELINE_ROWS = _employee_rows()

TASK_REGISTRY = {
    'schema_validation': {'name':'Customer Records Schema Validation','description':'Find all 13 issues in a 30-row CRM export.',
        'difficulty':'easy','max_steps':1,'task_type':'schema_validation',
        'rows':SCHEMA_VALIDATION_ROWS,'schema':SCHEMA_VALIDATION_SCHEMA,
        'known_issues':SCHEMA_VALIDATION_ISSUES,'sample_size':15},
    'standardization': {'name':'Sales Records Format Standardization','description':'Normalize 5 inconsistently-formatted columns.',
        'difficulty':'medium','max_steps':1,'task_type':'standardization',
        'rows':STANDARDIZATION_ROWS,'schema':{},'target_formats':STANDARDIZATION_TARGET_FORMATS,
        'ground_truth':STANDARDIZATION_GROUND_TRUTH,'sample_size':10},
    'pipeline': {'name':'Employee Records Quality Pipeline','description':'4-phase pipeline on 25-row employee data.',
        'difficulty':'hard','max_steps':8,'task_type':'pipeline',
        'rows':PIPELINE_ROWS,'schema':PIPELINE_SCHEMA,'known_issues':PIPELINE_ISSUES,
        'issue_categories':PIPELINE_ISSUE_CATEGORIES,'sample_size':15},
}
print(f'Datasets loaded: {[k for k in TASK_REGISTRY]} (issues: 13 / â€” / 10)')

## 4. Graders (Deterministic, 0.0 â€“ 1.0)

In [ ]:
import re

# â”€â”€ helpers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def _issue_key(i):   return (i['row_index'], i['column'], i['issue_type'])
def _pred_key(i):    return (i.row_index,    i.column,    i.issue_type.value)

_STATE_MAP = {'alabama':'AL','alaska':'AK','arizona':'AZ','arkansas':'AR','california':'CA',
              'colorado':'CO','connecticut':'CT','delaware':'DE','florida':'FL','georgia':'GA',
              'hawaii':'HI','idaho':'ID','illinois':'IL','indiana':'IN','iowa':'IA','kansas':'KS',
              'kentucky':'KY','louisiana':'LA','maine':'ME','maryland':'MD','massachusetts':'MA',
              'michigan':'MI','minnesota':'MN','mississippi':'MS','missouri':'MO','montana':'MT',
              'nebraska':'NE','nevada':'NV','new hampshire':'NH','new jersey':'NJ','new mexico':'NM',
              'new york':'NY','north carolina':'NC','north dakota':'ND','ohio':'OH','oklahoma':'OK',
              'oregon':'OR','pennsylvania':'PA','rhode island':'RI','south carolina':'SC',
              'south dakota':'SD','tennessee':'TN','texas':'TX','utah':'UT','vermont':'VT',
              'virginia':'VA','washington':'WA','west virginia':'WV','wisconsin':'WI','wyoming':'WY'}

# â”€â”€ Task 1: Schema Validation grader â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def grade_schema_validation(action, known_issues):
    if action.action_type != 'report_issues' or not action.issues:
        return RewardBreakdown(total=0.0, feedback='No issues reported.')
    gt   = {_issue_key(i) for i in known_issues}
    pred = {_pred_key(i)  for i in action.issues}
    exact_tp = len(gt & pred)
    gt_rc    = {(i['row_index'],i['column']) for i in known_issues}
    pd_rc    = {(i.row_index,i.column)       for i in action.issues}
    partial  = len((gt_rc & pd_rc) - {(k[0],k[1]) for k in (gt & pred)})
    eff_tp   = exact_tp + partial * 0.3
    prec     = eff_tp / len(pred) if pred else 0.0
    rec      = eff_tp / len(gt)   if gt   else 0.0
    f1       = 2*prec*rec/(prec+rec) if prec+rec else 0.0
    total    = min(f1, 1.0)
    if len(pred) > 2*len(gt): total *= 0.85
    return RewardBreakdown(total=round(total,4),
        components={'exact_tp':exact_tp,'partial':partial,'precision':round(prec,4),'recall':round(rec,4),'f1':round(f1,4)},
        feedback=f'GT:{len(gt)} Pred:{len(pred)} Exact:{exact_tp} Partial:{partial} F1:{f1:.3f}')

# â”€â”€ Task 2: Standardization grader â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def _norm_date(v, t):
    from datetime import datetime
    for fmt in ['%Y-%m-%d','%m/%d/%Y','%b %d %Y','%B %d %Y','%m/%d/%y','%d-%m-%Y','%m-%d-%Y']:
        try: return datetime.strptime(str(v).strip(), fmt).strftime('%Y-%m-%d')
        except: pass
def _norm_phone(v, t):
    d = re.sub(r'\D','',str(v))
    if d.startswith('1') and len(d)==11: d=d[1:]
    return f'({d[:3]}) {d[3:6]}-{d[6:]}' if len(d)==10 else None
def _norm_state(v, t):
    s=str(v).strip()
    if len(s)==2 and s.upper() in _STATE_MAP.values(): return s.upper()
    return _STATE_MAP.get(s.lower())
def _norm_amount(v, t):
    try: return round(float(re.sub(r'[^\d.]','',str(v).replace(',',''))),2)
    except: return None
def _norm_sku(v, t):
    d=re.sub(r'\D','',str(v))
    return f'SKU-{int(d):03d}' if d else None
_NORM = {'date':_norm_date,'phone':_norm_phone,'state':_norm_state,'amount':_norm_amount,'product_code':_norm_sku}

def grade_standardization(action, rows, ground_truth):
    if action.action_type != 'apply_transforms' or not action.transforms:
        return RewardBreakdown(total=0.0, feedback='No transforms.')
    n = len(rows); col_scores = {}
    for col in ground_truth:
        t = action.transforms.get(col)
        fn = _NORM.get(col)
        if not t or not fn: col_scores[col]=0.0; continue
        col_scores[col] = sum(1 for i,row in enumerate(rows)
                              if fn(str(row.get(col,'')),t)==ground_truth[col][i]) / n
    total = round(sum(col_scores.values())/len(col_scores),4)
    return RewardBreakdown(total=total,
        components={c:round(s,4) for c,s in col_scores.items()},
        feedback='Col scores: '+', '.join(f'{c}:{s:.0%}' for c,s in col_scores.items()))

# â”€â”€ Task 3: Pipeline graders â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def grade_pipeline_audit(action, known_cats):
    if action.action_type != 'audit':
        return RewardBreakdown(total=0.0, feedback='Need audit action.')
    txt = ' '.join([action.audit_summary or '', ' '.join(action.issue_categories or [])]).lower()
    found = [c for c in known_cats if c.replace('_',' ') in txt or c in txt]
    score = round(len(found)/len(known_cats),4) if known_cats else 0.0
    return RewardBreakdown(total=score,
        components={'found':len(found),'total':len(known_cats)},
        feedback=f'Found {len(found)}/{len(known_cats)} categories: {found}')

def grade_pipeline_identify(action, known_issues):
    return grade_schema_validation(action, known_issues)  # same F1 logic

def grade_pipeline_fix(action, known_issues):
    if action.action_type != 'fix' or not action.fix_operations:
        return RewardBreakdown(total=0.0, feedback='Need fix action.')
    locs = {(i['row_index'],i['column']) for i in known_issues}
    addressed = {(f.row_index,f.column) for f in action.fix_operations if (f.row_index,f.column) in locs}
    spurious  = sum(1 for f in action.fix_operations if (f.row_index,f.column) not in locs)
    rec = len(addressed)/len(known_issues) if known_issues else 0.0
    pen = min(spurious*0.05, 0.30)
    total = round(max(rec-pen,0.0),4)
    return RewardBreakdown(total=total,
        components={'addressed':len(addressed),'spurious':spurious,'recall':round(rec,4),'penalty':round(pen,4)},
        feedback=f'Fixed {len(addressed)}/{len(known_issues)}, {spurious} spurious, score={total}')

def grade_pipeline_validate(action, issues_fixed, total_issues):
    if action.action_type != 'validate':
        return RewardBreakdown(total=0.0, feedback='Need validate action.')
    rep = (action.validation_report or '').strip()
    s  = 0.4 if len(rep)>=30 else 0.0
    s += 0.3 if any(w in rep.lower() for w in ['issue','error','clean','valid','remain','fix']) else 0.0
    s += 0.15 if action.issues_remaining is not None else 0.0
    exp = total_issues - issues_fixed
    s += 0.15 if action.issues_remaining is not None and abs(action.issues_remaining-exp)<=2 else 0.0
    return RewardBreakdown(total=round(s,4), feedback=f'Report len:{len(rep)}, score:{s:.2f}')

def grade_pipeline_episode(phase_scores):
    w = {'audit':0.15,'identify':0.30,'fix':0.40,'validate':0.15}
    total = sum(w[p]*phase_scores.get(p,0.0) for p in w)
    if len(phase_scores)==4: total += 0.05
    return round(min(total,1.0),4)

print('Graders loaded.')

## 5. DataCleaningEnv â€” `reset()` Â· `step()` Â· `state()`

In [ ]:
import uuid

class _Episode:
    def __init__(self, episode_id, task_id):
        self.episode_id = episode_id; self.task_id = task_id
        self.task_cfg   = TASK_REGISTRY[task_id]
        self.task_type  = TaskType(self.task_cfg['task_type'])
        self.max_steps  = self.task_cfg['max_steps']
        self.current_step = 0; self.done = False
        self.cumulative_reward = 0.0; self._last_reward = 0.0
        self.action_history = []
        self.pipeline_phase = PipelineStep.AUDIT if self.task_type==TaskType.PIPELINE else None
        self.pipeline_scores = {}; self._fix_addressed = 0

def _col_stats(rows):
    if not rows: return {}
    stats = {}
    for col in rows[0]:
        vals = [r.get(col) for r in rows]
        non_null = [v for v in vals if v is not None]
        nums = [v for v in non_null if isinstance(v,(int,float))]
        stats[col] = ColumnStats(dtype=type(non_null[0]).__name__ if non_null else 'unknown',
            null_count=len(vals)-len(non_null), unique_count=len({str(v) for v in non_null}),
            sample_values=non_null[:5],
            min_value=min(nums) if nums else None, max_value=max(nums) if nums else None)
    return stats

def _avail_actions(ep):
    if ep.done: return []
    if ep.task_type==TaskType.SCHEMA_VALIDATION: return ['report_issues']
    if ep.task_type==TaskType.STANDARDIZATION:  return ['apply_transforms']
    return {PipelineStep.AUDIT:['audit'],PipelineStep.IDENTIFY:['report_issues'],
            PipelineStep.FIX:['fix'],PipelineStep.VALIDATE:['validate']}.get(ep.pipeline_phase,[])

def _build_obs(ep):
    cfg = ep.task_cfg; rows = cfg['rows']; n = cfg.get('sample_size',10)
    ctx = {}
    if ep.task_type==TaskType.STANDARDIZATION:
        ctx['target_formats']  = cfg.get('target_formats',{})
        ctx['columns_to_standardize'] = list(cfg.get('target_formats',{}).keys())
    if ep.task_type==TaskType.PIPELINE:
        ctx['current_phase']    = ep.pipeline_phase.value if ep.pipeline_phase else None
        ctx['phases_completed'] = list(ep.pipeline_scores.keys())
    return DataObservation(episode_id=ep.episode_id, task_type=ep.task_type,
        dataset_name=cfg['name'], dataset_sample=rows[:n], total_rows=len(rows),
        dataset_schema=cfg.get('schema',{}), column_stats=_col_stats(rows[:n]),
        current_step=ep.current_step, max_steps=ep.max_steps,
        pipeline_phase=ep.pipeline_phase, available_actions=_avail_actions(ep),
        reward=ep._last_reward, done=ep.done,
        cumulative_reward=ep.cumulative_reward, context=ctx)

class DataCleaningEnv:
    def __init__(self): self._episodes = {}

    def list_tasks(self):
        return [{'task_id':tid,'name':cfg['name'],'difficulty':cfg['difficulty'],
                 'max_steps':cfg['max_steps'],'task_type':cfg['task_type']}
                for tid,cfg in TASK_REGISTRY.items()]

    def reset(self, task_id='schema_validation'):
        if task_id not in TASK_REGISTRY: raise ValueError(f'Unknown task {task_id!r}')
        ep = _Episode(str(uuid.uuid4()), task_id)
        self._episodes[ep.episode_id] = ep
        return ResetResult(observation=_build_obs(ep), episode_id=ep.episode_id)

    def step(self, episode_id, action):
        ep = self._episodes.get(episode_id)
        if ep is None: raise KeyError(f'No episode {episode_id!r}')
        if ep.done:
            return StepResult(observation=_build_obs(ep), reward=0.0, done=True,
                              metadata={'error':'Episode done.'})
        ep.current_step += 1
        bd, done, next_phase = self._dispatch(ep, action)
        ep._last_reward = bd.total; ep.cumulative_reward += bd.total
        ep.action_history.append({'step':ep.current_step,'action':action.action_type,
                                   'reward':bd.total,'feedback':bd.feedback})
        if ep.task_type==TaskType.PIPELINE and next_phase: ep.pipeline_phase = next_phase
        if done or ep.current_step>=ep.max_steps: ep.done = True
        meta = {'reward_breakdown':bd.model_dump(),'step':ep.current_step}
        if ep.task_type==TaskType.PIPELINE and ep.done:
            meta['final_pipeline_score'] = grade_pipeline_episode(ep.pipeline_scores)
            meta['phase_scores']         = ep.pipeline_scores
        return StepResult(observation=_build_obs(ep), reward=bd.total, done=ep.done, metadata=meta)

    def state(self, episode_id):
        ep = self._episodes.get(episode_id)
        if ep is None: raise KeyError(f'No episode {episode_id!r}')
        return StateResult(episode_id=ep.episode_id, task_type=ep.task_type,
            current_step=ep.current_step, max_steps=ep.max_steps,
            pipeline_phase=ep.pipeline_phase, done=ep.done,
            cumulative_reward=ep.cumulative_reward, action_history=ep.action_history)

    def _dispatch(self, ep, action):
        cfg = ep.task_cfg
        if ep.task_type==TaskType.SCHEMA_VALIDATION:
            return grade_schema_validation(action,cfg['known_issues']), True, None
        if ep.task_type==TaskType.STANDARDIZATION:
            return grade_standardization(action,cfg['rows'],cfg['ground_truth']), True, None
        # pipeline
        ph = ep.pipeline_phase
        if ph==PipelineStep.AUDIT:
            bd=grade_pipeline_audit(action,cfg['issue_categories'])
            ep.pipeline_scores['audit']=bd.total; return bd, False, PipelineStep.IDENTIFY
        if ph==PipelineStep.IDENTIFY:
            bd=grade_pipeline_identify(action,cfg['known_issues'])
            ep.pipeline_scores['identify']=bd.total; return bd, False, PipelineStep.FIX
        if ph==PipelineStep.FIX:
            bd=grade_pipeline_fix(action,cfg['known_issues'])
            ep.pipeline_scores['fix']=bd.total; ep._fix_addressed=bd.components.get('addressed',0)
            return bd, False, PipelineStep.VALIDATE
        # validate
        bd=grade_pipeline_validate(action,ep._fix_addressed,len(cfg['known_issues']))
        ep.pipeline_scores['validate']=bd.total; return bd, True, None

env = DataCleaningEnv()
print('Environment ready. Tasks:', [t['task_id'] for t in env.list_tasks()])

## 6. Task 1 â€” Schema Validation (Easy)
Agent finds all 13 issues in a 30-row customer CRM export. Scored by F1 over `(row, column, issue_type)` tuples.

In [ ]:
r = env.reset('schema_validation')
print(f'Episode: {r.episode_id[:8]}...')
print(f'Dataset: {r.observation.dataset_name} ({r.observation.total_rows} rows)')
print(f'Schema fields: {list(r.observation.dataset_schema.keys())}')
print(f'Sample row 0: {r.observation.dataset_sample[0]}')
print(f'Available actions: {r.observation.available_actions}')

In [ ]:
# Perfect answer â€” agent reports all 13 ground-truth issues
perfect_action = DataAction(
    action_type='report_issues',
    issues=[DataIssue(**{**i, 'issue_type': IssueType(i['issue_type'])}) for i in SCHEMA_VALIDATION_ISSUES]
)
sr = env.step(r.episode_id, perfect_action)
print(f'Reward: {sr.reward:.4f}  Done: {sr.done}')
print(f'Obs.reward: {sr.observation.reward:.4f}  Obs.done: {sr.observation.done}')
print(f'Feedback: {sr.metadata["reward_breakdown"]["feedback"]}')

In [ ]:
# Partial answer â€” agent finds 5 of 13 issues
r2 = env.reset('schema_validation')
partial_action = DataAction(
    action_type='report_issues',
    issues=[DataIssue(**{**i, 'issue_type': IssueType(i['issue_type'])}) for i in SCHEMA_VALIDATION_ISSUES[:5]]
)
sr2 = env.step(r2.episode_id, partial_action)
print(f'Partial (5/13 issues)  Reward: {sr2.reward:.4f}')
bd = sr2.metadata['reward_breakdown']['components']
print(f'  Precision: {bd["precision"]:.2f}  Recall: {bd["recall"]:.2f}  F1: {bd["f1"]:.2f}')

## 7. Task 2 â€” Format Standardization (Medium)
Agent defines transformation rules for 5 inconsistently-formatted columns.
Rules are applied programmatically and compared to ground-truth expected outputs.

In [ ]:
import pandas as pd

r = env.reset('standardization')
print('Target formats:', r.observation.context.get('target_formats'))
print()
# Show the messy data
df = pd.DataFrame(r.observation.dataset_sample)
df[['date','phone','state','amount','product_code']].head(8)

In [ ]:
# Agent submits transformation rules
action = DataAction(
    action_type='apply_transforms',
    transforms={
        'date':         ColumnTransform(target_format='ISO 8601: YYYY-MM-DD',
                            examples=[{'before':'01/15/2023','after':'2023-01-15'},
                                      {'before':'Mar 10 2023','after':'2023-03-10'}]),
        'phone':        ColumnTransform(target_format='(XXX) XXX-XXXX',
                            examples=[{'before':'5551234567','after':'(555) 123-4567'},
                                      {'before':'+15551234567','after':'(555) 123-4567'}]),
        'state':        ColumnTransform(target_format='Two-letter uppercase (CA, NY ...)',
                            examples=[{'before':'California','after':'CA'},
                                      {'before':'ohio','after':'OH'}]),
        'amount':       ColumnTransform(target_format='Plain float (1234.56)',
                            examples=[{'before':'$1,234.56','after':'1234.56'},
                                      {'before':'USD 2500.00','after':'2500.0'}]),
        'product_code': ColumnTransform(target_format='SKU-NNN zero-padded',
                            examples=[{'before':'sku001','after':'SKU-001'},
                                      {'before':'003','after':'SKU-003'}]),
    }
)
sr = env.step(r.episode_id, action)
print(f'Reward: {sr.reward:.4f}')
print('Per-column scores:')
for col, score in sr.metadata['reward_breakdown']['components'].items():
    bar = 'â–ˆ' * int(score*20) + 'â–‘'*(20-int(score*20))
    print(f'  {col:<15} {bar} {score:.1%}')

## 8. Task 3 â€” Multi-step Pipeline (Hard)
Four phases must be completed in sequence: **Audit â†’ Identify â†’ Fix â†’ Validate**.
Each phase is scored independently; a +0.05 efficiency bonus applies if all 4 complete.

In [ ]:
import pandas as pd

r = env.reset('pipeline')
print(f'Phase: {r.observation.pipeline_phase}  Available: {r.observation.available_actions}')
print()
df = pd.DataFrame(r.observation.dataset_sample)
# Highlight columns with issues
df[['employee_id','full_name','department','salary','hire_date','vacation_days','email']].head(10)

In [ ]:
eid = r.episode_id

# Phase 1: AUDIT â€” identify issue categories (no specific rows yet)
sr1 = env.step(eid, DataAction(
    action_type='audit',
    audit_summary='Dataset contains salary outliers, duplicate employee IDs, '
                  'invalid department enums, missing required fields, '
                  'non-ISO hire_date formats, negative vacation_days, and a bad email.',
    issue_categories=PIPELINE_ISSUE_CATEGORIES
))
print(f'AUDIT    reward={sr1.reward:.4f}  next={sr1.observation.pipeline_phase}')
print(f'         {sr1.metadata["reward_breakdown"]["feedback"]}')

In [ ]:
# Phase 2: IDENTIFY â€” enumerate every specific issue
sr2 = env.step(eid, DataAction(
    action_type='report_issues',
    issues=[DataIssue(**{**i, 'issue_type': IssueType(i['issue_type'])}) for i in PIPELINE_ISSUES]
))
print(f'IDENTIFY reward={sr2.reward:.4f}  next={sr2.observation.pipeline_phase}')
print(f'         {sr2.metadata["reward_breakdown"]["feedback"]}')

In [ ]:
# Phase 3: FIX â€” apply corrections
fixes = [FixOperation(row_index=i['row_index'], column=i['column'],
                      old_value=PIPELINE_ROWS[i['row_index']].get(i['column']),
                      new_value='CORRECTED', rationale='fix')
         for i in PIPELINE_ISSUES]
sr3 = env.step(eid, DataAction(action_type='fix', fix_operations=fixes))
print(f'FIX      reward={sr3.reward:.4f}  next={sr3.observation.pipeline_phase}')
print(f'         {sr3.metadata["reward_breakdown"]["feedback"]}')

In [ ]:
# Phase 4: VALIDATE â€” confirm data quality
sr4 = env.step(eid, DataAction(
    action_type='validate',
    validation_report='All 10 identified issues have been corrected. '
                      'Salary outliers capped, duplicate ID resolved, '
                      'dates normalised, negative vacation_days set to 0, '
                      'department enum fixed, email corrected. Dataset is clean.',
    issues_remaining=0
))
print(f'VALIDATE reward={sr4.reward:.4f}  done={sr4.done}')
print()
print('â”€'*50)
print(f'Final pipeline score : {sr4.metadata["final_pipeline_score"]}')
print(f'Phase breakdown      : {sr4.metadata["phase_scores"]}')

## 9. FastAPI Server â€” OpenEnv HTTP Spec
Starts the server in a background thread (Colab-compatible). Exposes:
`/reset` Â· `/step` Â· `/state/{id}` Â· `/health` Â· `/schema` Â· `/metadata` Â· `/validate`

In [ ]:
import threading, time, requests as _req
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel as _BM
import uvicorn

_app = FastAPI(title='Data Cleaning OpenEnv', version='1.0.0')
_app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])
_env = DataCleaningEnv()

class _ResetReq(_BM): task_id: str = 'schema_validation'
class _StepReq(_BM):  episode_id: str; action: DataAction

@_app.get('/')        
def _root():          return {'name':'data-cleaning-env','status':'healthy','openenv_spec':'1.0'}
@_app.get('/health')  
def _health():        return {'status':'healthy','version':'1.0.0'}
@_app.get('/tasks')   
def _tasks():         return {'tasks': _env.list_tasks()}
@_app.get('/metadata')
def _metadata():      return {'name':'data-cleaning-env','version':'1.0.0','tasks':_env.list_tasks()}
@_app.get('/schema')  
def _schema():        return {'action':DataAction.model_json_schema(),'observation':DataObservation.model_json_schema()}
@_app.post('/reset')  
def _reset(req: _ResetReq):
    try:    return _env.reset(task_id=req.task_id)
    except ValueError as e: raise HTTPException(400, str(e))
@_app.post('/step')
def _step(req: _StepReq):
    try:    return _env.step(req.episode_id, req.action)
    except KeyError as e:   raise HTTPException(404, str(e))
@_app.get('/state/{eid}')
def _state(eid: str):
    try:    return _env.state(eid)
    except KeyError as e:   raise HTTPException(404, str(e))

PORT = 7860
server_thread = threading.Thread(
    target=lambda: uvicorn.run(_app, host='0.0.0.0', port=PORT, log_level='error'),
    daemon=True
)
server_thread.start()
time.sleep(3)

r = _req.get(f'http://localhost:{PORT}/health')
print('Server status:', r.json())

In [ ]:
# HTTP round-trip test: reset â†’ step â†’ state
BASE = f'http://localhost:{PORT}'

rr = _req.post(f'{BASE}/reset', json={'task_id':'schema_validation'}).json()
eid = rr['episode_id']
print(f'reset   episode={eid[:8]}  obs.reward={rr["observation"]["reward"]}  obs.done={rr["observation"]["done"]}')

sr = _req.post(f'{BASE}/step', json={
    'episode_id': eid,
    'action': {'action_type':'report_issues', 'issues':[
        {'row_index':2,  'column':'email',  'issue_type':'invalid_format',   'description':'bad','value':'x'},
        {'row_index':6,  'column':'age',    'issue_type':'invalid_range',    'description':'neg','value':-5},
        {'row_index':9,  'column':'id',     'issue_type':'duplicate',        'description':'dup','value':2},
        {'row_index':11, 'column':'email',  'issue_type':'missing_required', 'description':'null','value':None},
    ]}
}).json()
print(f'step    reward={sr["reward"]:.4f}  done={sr["done"]}')
print(f'        obs.reward={sr["observation"]["reward"]:.4f}  metadata keys={list(sr["metadata"].keys())}')

st = _req.get(f'{BASE}/state/{eid}').json()
print(f'state   step={st["current_step"]}  cumulative_reward={st["cumulative_reward"]:.4f}')

## 10. LLM Baseline Inference
Uses the OpenAI-compatible client to run a model against all 3 tasks.

Set your credentials below, or pass them as Colab secrets.

In [ ]:
import os, json, re, textwrap
from openai import OpenAI

# â”€â”€ Credentials (edit or use Colab Secrets) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
API_BASE_URL = os.getenv('API_BASE_URL', 'https://router.huggingface.co/v1')
MODEL_NAME   = os.getenv('MODEL_NAME',   'Qwen/Qwen2.5-72B-Instruct')
API_KEY      = os.getenv('HF_TOKEN') or os.getenv('API_KEY', '')

client = OpenAI(base_url=API_BASE_URL, api_key=API_KEY)
print(f'Using model: {MODEL_NAME}')
print(f'API base:    {API_BASE_URL}')

In [ ]:
def _extract_json(text):
    text = re.sub(r'```(?:json)?\s*', '', text, flags=re.I).replace('```','').strip()
    start = text.find('{')
    if start == -1: return None
    depth = 0
    for i, ch in enumerate(text[start:], start):
        depth += (ch=='{') - (ch=='}')
        if depth == 0:
            try: return json.loads(text[start:i+1])
            except: return None

def _call(system, user, max_tokens=1200):
    try:
        r = client.chat.completions.create(model=MODEL_NAME, temperature=0.1,
            max_tokens=max_tokens,
            messages=[{'role':'system','content':system},{'role':'user','content':user}])
        return r.choices[0].message.content or ''
    except Exception as e:
        print(f'  [LLM error] {e}'); return ''

def _obs_text(obs_dict):
    lines = [f"Task: {obs_dict.get('task_type')} | {obs_dict.get('dataset_name')}",
             f"Step: {obs_dict.get('current_step')}/{obs_dict.get('max_steps')}"]
    if obs_dict.get('pipeline_phase'):
        lines.append(f"Current phase: {obs_dict['pipeline_phase']}")
    ctx = obs_dict.get('context',{})
    if ctx.get('target_formats'):
        lines.append('Target formats: '+json.dumps(ctx['target_formats']))
    if obs_dict.get('dataset_schema'):
        lines.append('Schema: '+json.dumps(obs_dict['dataset_schema']))
    lines.append('Sample data: '+json.dumps(obs_dict.get('dataset_sample',[])[:8], default=str))
    lines.append('Column stats: '+json.dumps({k:{'null':v['null_count'],'samples':v['sample_values']}
        for k,v in obs_dict.get('column_stats',{}).items()}, default=str))
    return '\n'.join(lines)

print('Helpers ready.')

In [ ]:
# â”€â”€ Task 1: Schema Validation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
SYSTEM_1 = textwrap.dedent('''
You are a data quality expert. Identify ALL issues in this dataset against the schema.
Valid issue_type values: missing_required, wrong_type, invalid_range, invalid_format, invalid_enum, duplicate, outlier
row_index is 0-based. For duplicates report the later row.
Respond ONLY with this JSON:
{"action_type": "report_issues",
 "issues": [{"row_index": N, "column": "...", "issue_type": "...", "description": "...", "value": ...}]}
''').strip()

rr   = _req.post(f'{BASE}/reset', json={'task_id':'schema_validation'}).json()
raw  = _call(SYSTEM_1, _obs_text(rr['observation']))
act  = _extract_json(raw) or {'action_type':'report_issues','issues':[]}
sr   = _req.post(f'{BASE}/step', json={'episode_id':rr['episode_id'],'action':act}).json()
bd   = sr['metadata']['reward_breakdown']
print(f'Task 1 â€“ Schema Validation')
print(f'  Issues reported: {len(act.get("issues",[])):2d} | Score: {sr["reward"]:.4f}')
print(f'  {bd["feedback"]}')

In [ ]:
# â”€â”€ Task 2: Standardization â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
SYSTEM_2 = textwrap.dedent('''
You are a data engineer. Define transformation rules to normalize each listed column.
Respond ONLY with this JSON:
{"action_type": "apply_transforms",
 "transforms": {
   "<column>": {"target_format": "...", "regex_from": null, "replace_with": null,
                "examples": [{"before": "...", "after": "..."}]}
 }}
Include an entry for EVERY column in context.columns_to_standardize.
''').strip()

rr   = _req.post(f'{BASE}/reset', json={'task_id':'standardization'}).json()
raw  = _call(SYSTEM_2, _obs_text(rr['observation']))
act  = _extract_json(raw) or {'action_type':'apply_transforms','transforms':{}}
sr   = _req.post(f'{BASE}/step', json={'episode_id':rr['episode_id'],'action':act}).json()
bd   = sr['metadata']['reward_breakdown']
print(f'Task 2 â€“ Standardization')
print(f'  Columns covered: {list(act.get("transforms",{}).keys())} | Score: {sr["reward"]:.4f}')
print(f'  {bd["feedback"]}')

In [ ]:
# â”€â”€ Task 3: Pipeline â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
SYSTEM_PIPELINE = {
  'audit':    'You are auditing a dataset. Identify issue CATEGORIES (not specific rows). '
              'Valid: missing_required, wrong_type, invalid_range, invalid_format, invalid_enum, duplicate, outlier. '
              'Respond ONLY: {"action_type":"audit","audit_summary":"...","issue_categories":[...]}',
  'identify': 'Enumerate EVERY specific issue. row_index is 0-based. '
              'Respond ONLY: {"action_type":"report_issues","issues":[{"row_index":N,"column":"...","issue_type":"...","description":"...","value":...}]}',
  'fix':      'Apply a fix_operation for each issue. '
              'Respond ONLY: {"action_type":"fix","fix_operations":[{"row_index":N,"column":"...","old_value":...,"new_value":...,"rationale":"..."}]}',
  'validate': 'Write a validation report and estimate remaining issues. '
              'Respond ONLY: {"action_type":"validate","validation_report":"...","issues_remaining":N}',
}
FALLBACKS = {
  'audit':    {'action_type':'audit','audit_summary':'No issues.','issue_categories':[]},
  'identify': {'action_type':'report_issues','issues':[]},
  'fix':      {'action_type':'fix','fix_operations':[]},
  'validate': {'action_type':'validate','validation_report':'Done.','issues_remaining':0},
}

rr  = _req.post(f'{BASE}/reset', json={'task_id':'pipeline'}).json()
eid = rr['episode_id']; obs = rr['observation']
phase_results = {}

for _ in range(8):
    phase = obs.get('pipeline_phase')
    if not phase: break
    raw = _call(SYSTEM_PIPELINE[phase], _obs_text(obs))
    act = _extract_json(raw) or FALLBACKS[phase]
    sr  = _req.post(f'{BASE}/step', json={'episode_id':eid,'action':act}).json()
    phase_results[phase] = sr['reward']
    print(f'  {phase:<10} reward={sr["reward"]:.4f}  {sr["metadata"]["reward_breakdown"]["feedback"][:80]}')
    obs = sr['observation']
    if sr['done']: break

final = sr['metadata'].get('final_pipeline_score', 0.0)
print(f'\nTask 3 â€“ Pipeline  Final score: {final:.4f}  Phase breakdown: {phase_results}')

In [ ]:
# â”€â”€ Summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
scores = {
    'schema_validation (easy)':   sr['reward'] if False else None,  # set from cells above
}
# Re-run all 3 tasks cleanly and collect scores
def _run_all():
    results = {}

    # T1
    rr = _req.post(f'{BASE}/reset', json={'task_id':'schema_validation'}).json()
    raw = _call(SYSTEM_1, _obs_text(rr['observation']))
    act = _extract_json(raw) or {'action_type':'report_issues','issues':[]}
    sr  = _req.post(f'{BASE}/step', json={'episode_id':rr['episode_id'],'action':act}).json()
    results['schema_validation (easy)'] = sr['reward']

    # T2
    rr = _req.post(f'{BASE}/reset', json={'task_id':'standardization'}).json()
    raw = _call(SYSTEM_2, _obs_text(rr['observation']))
    act = _extract_json(raw) or {'action_type':'apply_transforms','transforms':{}}
    sr  = _req.post(f'{BASE}/step', json={'episode_id':rr['episode_id'],'action':act}).json()
    results['standardization (medium)'] = sr['reward']

    # T3
    rr = _req.post(f'{BASE}/reset', json={'task_id':'pipeline'}).json()
    eid = rr['episode_id']; obs = rr['observation']
    for _ in range(8):
        phase = obs.get('pipeline_phase')
        if not phase: break
        raw = _call(SYSTEM_PIPELINE[phase], _obs_text(obs))
        act = _extract_json(raw) or FALLBACKS[phase]
        sr  = _req.post(f'{BASE}/step', json={'episode_id':eid,'action':act}).json()
        obs = sr['observation']
        if sr['done']: break
    results['pipeline (hard)'] = sr['metadata'].get('final_pipeline_score', 0.0)

    return results

print('Running baseline (this makes LLM API calls)...')
scores = _run_all()

print()
print('='*55)
print('  BASELINE SCORES')
print('='*55)
for task, score in scores.items():
    bar = 'â–ˆ'*int(score*25) + 'â–‘'*(25-int(score*25))
    print(f'  {task:<30} {bar} {score:.4f}')
overall = sum(scores.values())/len(scores)
print(f'  {"Overall average":<30} {"â”€"*25} {overall:.4f}')
print('='*55)